# RakForest — ทบทวนคลิปเสียงที่อาจติดป้ายผิด

ฟังเสียงจาก `zenodo_fire` ทีละช่วง 4 วินาที และบันทึกทุกคลาสที่ได้ยินร่วมกันได้ เช่น `fire` พร้อม `human_activity`. Zenodo Fire ยังคงถูกกักออกจากชุดฝึกจนกว่าจะตรวจและตัดสินใจอย่างชัดเจน.

- อ่าน raw แบบอ่านอย่างเดียว ไม่แก้ checkpoint หรือ processed manifest
- ผลตรวจแยกเก็บใน `reports/zenodo_fire_segment_review.csv`
- การสร้างรายการช่วงอ่านเพียง header ของ WAV; player อ่านเฉพาะช่วงที่เลือก

## วิธีทำงานของไฟล์นี้: ให้คนฟังตรวจป้ายเสียง

ไฟล์นี้เป็นขั้น **human review / ตรวจคุณภาพข้อมูล** ไม่ได้ฝึกหรือแก้ checkpoint อัตโนมัติ สร้างรายการช่วงเสียงให้ฟัง แล้วให้ผู้ตรวจบันทึกคลาสที่ได้ยินร่วมกันใน `present_labels` พร้อมสถานะเสียงไฟ, เสียงคน/คำพูด, คุณภาพช่วง และหมายเหตุ

ใช้บันทึกนี้ค้นหาป้ายที่ผิด เสียงเงียบ/ว่าง หรือเหตุการณ์หลายชนิดที่เกิดพร้อมกัน ก่อนตัดสินใจปรับ metadata หรือสร้างชุดฝึกใหม่ การฟังตรวจเป็นหลักฐานช่วยประเมิน weak labels และเสียงผสมสังเคราะห์ แต่ผลในไฟล์ review จะไม่เปลี่ยน manifest หรือโมเดลเอง ต้องนำผลไปปรับข้อมูลและรัน pipeline ที่เกี่ยวข้องภายหลัง


## 1. Imports และตำแหน่งข้อมูล

In [15]:
import importlib.util, subprocess, sys
required={'numpy':'numpy','pandas':'pandas','soundfile':'soundfile','IPython':'ipython'}
missing=[package for module,package in required.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install',*missing])

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import Audio, display, Markdown
from tqdm.auto import tqdm

PROJECT_ROOT=Path.cwd()
if PROJECT_ROOT.name=='notebooks': PROJECT_ROOT=PROJECT_ROOT.parent
SOURCE_DIR=PROJECT_ROOT/'data'/'raw'/'zenodo_fire'
REPORT_DIR=PROJECT_ROOT/'reports'; REPORT_DIR.mkdir(parents=True,exist_ok=True)
REVIEW_PATH=REPORT_DIR/'zenodo_fire_segment_review.csv'
SEGMENT_SECONDS=4.0
AUDIO_EXTENSIONS={'.wav','.wave','.flac','.ogg','.mp3','.m4a'}
print('Project:',PROJECT_ROOT)
print('Source:',SOURCE_DIR)
print('Review output:',REVIEW_PATH)

Project: /Users/guidegiegg/Projects/RakForest-ML
Source: /Users/guidegiegg/Projects/RakForest-ML/data/raw/zenodo_fire
Review output: /Users/guidegiegg/Projects/RakForest-ML/reports/zenodo_fire_segment_review.csv


## 2. สร้างบัญชีช่วงเสียงสำหรับฟัง

In [16]:
if not SOURCE_DIR.exists(): raise FileNotFoundError(f'ไม่พบโฟลเดอร์ {SOURCE_DIR}')
source_files=sorted(p for p in SOURCE_DIR.rglob('*') if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS and not p.name.startswith('.'))
if not source_files: raise RuntimeError('ไม่พบไฟล์เสียงใน Zenodo Fire')
rows=[]
for path in tqdm(source_files,desc='Read audio headers'):
    try:
        info=sf.info(path)
        n_segments=max(1,int(np.ceil(info.duration/SEGMENT_SECONDS)))
        for ix in range(n_segments):
            start=ix*SEGMENT_SECONDS; end=min((ix+1)*SEGMENT_SECONDS,float(info.duration))
            rel=path.relative_to(PROJECT_ROOT).as_posix()
            source_id=hashlib.sha1(rel.encode('utf-8')).hexdigest()[:10]
            rows.append({'segment_id':f'{source_id}__{ix:05d}','source_file':rel,'segment_index':ix,'start_seconds':start,'end_seconds':end,'duration_seconds':end-start,'sample_rate':info.samplerate,'channels':info.channels})
    except Exception as e:
        print(f'[skip unreadable header] {path.name}: {e}')
segments=pd.DataFrame(rows)
if segments.empty: raise RuntimeError('สร้างรายการช่วงเสียงไม่ได้')
print(f'{len(source_files):,} source files; {len(segments):,} segments of up to {SEGMENT_SECONDS:g}s')
display(segments.head())

Read audio headers:   0%|          | 0/47 [00:00<?, ?it/s]

47 source files; 7,678 segments of up to 4s


,segment_id,source_file,segment_index,start_seconds,end_seconds,duration_seconds,sample_rate,channels
0,37a19deca8__00000,data/raw/zenodo_fire/extracted/wav/0exp1b.WAV,0,0.0,4.0,4.0,96000,1
1,37a19deca8__00001,data/raw/zenodo_fire/extracted/wav/0exp1b.WAV,1,4.0,8.0,4.0,96000,1
2,37a19deca8__00002,data/raw/zenodo_fire/extracted/wav/0exp1b.WAV,2,8.0,12.0,4.0,96000,1
3,37a19deca8__00003,data/raw/zenodo_fire/extracted/wav/0exp1b.WAV,3,12.0,16.0,4.0,96000,1
4,37a19deca8__00004,data/raw/zenodo_fire/extracted/wav/0exp1b.WAV,4,16.0,20.0,4.0,96000,1


## 3. โหลดผลตรวจเดิมและเลือกช่วงที่จะฟัง

In [17]:
import json
REVIEW_FIELDS=['fire_audible','speech_audible','present_labels','segment_quality','notes']
if REVIEW_PATH.exists():
    old=pd.read_csv(REVIEW_PATH)
    keep=[c for c in ['segment_id']+REVIEW_FIELDS if c in old.columns]
    segments=segments.merge(old[keep].drop_duplicates('segment_id'),on='segment_id',how='left')
else:
    for col in REVIEW_FIELDS: segments[col]=pd.NA

def is_review_complete(row):
    # "unsure", blank answers, and an empty tag list remain pending for another listen.
    def text_value(value): return str(value).strip().lower() if pd.notna(value) else ''
    if text_value(row.get('fire_audible')) not in {'yes','no'}: return False
    if text_value(row.get('speech_audible')) not in {'yes','no'}: return False
    if text_value(row.get('segment_quality')) in {'','unsure','nan','none'}: return False
    raw=row.get('present_labels')
    if pd.isna(raw): return False
    try: labels=json.loads(raw) if isinstance(raw,str) else list(raw)
    except Exception: return False
    return isinstance(labels,list) and len(labels)>0

# Incomplete or unsure rows stay in the queue. REVIEW_ROW selects one pending segment.
pending_mask=~segments.apply(is_review_complete,axis=1)
unreviewed=segments[pending_mask].reset_index(drop=True)
REVIEW_ROW=0
if unreviewed.empty:
    print('ตรวจครบทุกช่วงแล้ว')
else:
    if not 0 <= REVIEW_ROW < len(unreviewed): raise IndexError(f'REVIEW_ROW ต้องอยู่ระหว่าง 0 ถึง {len(unreviewed)-1}')
    current=unreviewed.iloc[REVIEW_ROW]
    current_path=PROJECT_ROOT/current.source_file
    with sf.SoundFile(current_path) as f:
        f.seek(int(current.start_seconds*f.samplerate))
        audio=f.read(int(current.duration_seconds*f.samplerate),dtype='float32',always_2d=True)
    audio=audio.mean(axis=1)
    display(Markdown(f"### {current.segment_id} — {current.start_seconds:.1f}–{current.end_seconds:.1f}s\nไฟล์: `{current.source_file}` | ช่วงรอตรวจลำดับ {REVIEW_ROW+1:,}/{len(unreviewed):,}"))
    display(Audio(audio,rate=int(current.sample_rate)))


### 37a19deca8__00000 — 0.0–4.0s
ไฟล์: `data/raw/zenodo_fire/extracted/wav/0exp1b.WAV` | ช่วงรอตรวจลำดับ 1/7,678

## 4. บันทึกผลตรวจช่วงที่กำลังฟัง
ระบุคลาสทุกชนิดที่ได้ยินใน `PRESENT_LABELS` เช่น `['fire','human_activity','weather']`. ใช้ `silence` หรือ `other` แยกเดี่ยวเท่านั้น. ค่านี้เป็น annotation สำหรับ review ไม่ได้ไหลเข้า manifest หรือ train อัตโนมัติ.

ถ้าเลือก `unsure` หรือยังไม่ใส่ `PRESENT_LABELS` ผลจะถูกบันทึกไว้ แต่ช่วงนั้นจะยังอยู่ในคิวให้กลับมาฟังซ้ำ; ช่วงจะนับว่าเสร็จเมื่อ fire/speech ตอบ yes/no, quality ชัดเจน และระบุอย่างน้อยหนึ่งคลาสที่ได้ยินแล้วเท่านั้น.


In [18]:
import json  # required to serialize the multi-label review tags
# กรอกตามที่ได้ยิน: fire_audible / speech_audible = yes, no, unsure
# segment_quality = usable_fire, mixed_fire_and_speech, speech_or_human_only, silence, other, unusable, unsure
# ใส่ชื่อผู้ตรวจ/รายละเอียดเพิ่มใน notes ได้
FIRE_AUDIBLE='unsure'
SPEECH_AUDIBLE='unsure'
SEGMENT_QUALITY='unsure'
NOTES=''
REVIEWER=''
PRESENT_LABELS=[]  # ตัวอย่าง: ['fire','human_activity','weather']

valid={'yes','no','unsure'}
quality={'usable_fire','mixed_fire_and_speech','speech_or_human_only','silence','other','unusable','unsure'}
valid_labels={'fire','chainsaw','vehicle','human_activity','threat_tactical','weather','natural_ambient','silence','other'}
if not unreviewed.empty:
    if FIRE_AUDIBLE not in valid or SPEECH_AUDIBLE not in valid: raise ValueError(f'ค่าการได้ยินต้องเป็น {sorted(valid)}')
    if SEGMENT_QUALITY not in quality: raise ValueError(f'segment quality ต้องเป็น {sorted(quality)}')
    unknown=set(PRESENT_LABELS)-valid_labels
    if unknown: raise ValueError(f'คลาสไม่รู้จัก: {sorted(unknown)}')
    if len(PRESENT_LABELS)!=len(set(PRESENT_LABELS)): raise ValueError('มีคลาสซ้ำใน PRESENT_LABELS')
    if 'silence' in PRESENT_LABELS and len(PRESENT_LABELS)>1: raise ValueError('silence ต้องเป็นแท็กเดี่ยว')
    if 'other' in PRESENT_LABELS and len(PRESENT_LABELS)>1: raise ValueError('other ต้องเป็นแท็กเดี่ยว')
    if FIRE_AUDIBLE!='unsure' and SPEECH_AUDIBLE!='unsure' and SEGMENT_QUALITY!='unsure' and not PRESENT_LABELS: raise ValueError('กรอก PRESENT_LABELS อย่างน้อย 1 คลาส; silence/other ใช้แท็กเดี่ยว')
    record=current[['segment_id','source_file','segment_index','start_seconds','end_seconds']].to_dict()
    record.update(fire_audible=FIRE_AUDIBLE,speech_audible=SPEECH_AUDIBLE,present_labels=json.dumps(PRESENT_LABELS,ensure_ascii=False),segment_quality=SEGMENT_QUALITY,notes=NOTES,reviewer=REVIEWER)
    previous=pd.read_csv(REVIEW_PATH) if REVIEW_PATH.exists() else pd.DataFrame()
    updated=pd.concat([previous,pd.DataFrame([record])],ignore_index=True)
    updated=updated.drop_duplicates('segment_id',keep='last').sort_values('segment_id')
    updated.to_csv(REVIEW_PATH,index=False)
    print('Saved review:',current.segment_id,'→',REVIEW_PATH)
else:
    print('ไม่มีช่วงใหม่ให้บันทึก')

Saved review: 37a19deca8__00000 → /Users/guidegiegg/Projects/RakForest-ML/reports/zenodo_fire_segment_review.csv


## 5. สรุปและส่งออกคิวตรวจ

In [19]:
if REVIEW_PATH.exists():
    reviewed=pd.read_csv(REVIEW_PATH)
    complete_mask=reviewed.apply(is_review_complete,axis=1)
    completed_ids=set(reviewed.loc[complete_mask,'segment_id'].astype(str))
    print(f'ตรวจครบแล้ว {len(completed_ids):,} / {len(segments):,} segments')
    print(f'มีแถวที่บันทึกไว้แต่ยังไม่สมบูรณ์/ยัง unsure: {int((~complete_mask).sum()):,}')
    if not reviewed.empty:
        display(reviewed.groupby(['segment_quality','fire_audible','speech_audible'],dropna=False).size().rename('saved rows').reset_index())
        if 'present_labels' in reviewed:
            display(reviewed.groupby('present_labels',dropna=False).size().rename('saved rows').reset_index().sort_values('saved rows',ascending=False))
    remaining=segments[~segments.segment_id.astype(str).isin(completed_ids)]
else:
    reviewed=pd.DataFrame(); completed_ids=set(); remaining=segments
queue_path=REPORT_DIR/'zenodo_fire_review_queue.csv'
remaining.to_csv(queue_path,index=False)
print(f'ช่วงที่ยังรอตรวจ/ยืนยัน: {len(remaining):,}')
print('Remaining review queue:',queue_path)
print('Review annotations:',REVIEW_PATH)


ตรวจครบแล้ว 0 / 7,678 segments
มีแถวที่บันทึกไว้แต่ยังไม่สมบูรณ์/ยัง unsure: 11


,segment_quality,fire_audible,speech_audible,saved rows
0,unsure,unsure,unsure,11


,present_labels,saved rows
1,NaN,9
0,[],2


ช่วงที่ยังรอตรวจ/ยืนยัน: 7,678
Remaining review queue: /Users/guidegiegg/Projects/RakForest-ML/reports/zenodo_fire_review_queue.csv
Review annotations: /Users/guidegiegg/Projects/RakForest-ML/reports/zenodo_fire_segment_review.csv


## วิธีใช้ผลตรวจ

ตรวจทีละช่วงโดยตั้ง `REVIEW_ROW` แล้วรันเซลล์ 3 และ 4; การบันทึกครั้งใหม่จะอัปเดตแถวเดิมของช่วงนั้น ไม่สร้างซ้ำ หลังตรวจแล้ว ใช้สรุปผลตัดสินใจว่าจะคง Zenodo Fire ไว้ใน quarantine, คัดเฉพาะช่วงที่เป็นไฟจริง หรือไม่นำชุดนี้มาใช้

ไฟล์ annotation นี้ยังไม่เปลี่ยนชุดฝึกโดยอัตโนมัติ เพราะควรตรวจความครบถ้วนและ label ด้วยคนก่อน และไม่ควรใช้ผล test มาปรับเกณฑ์ของโมเดล